# 실험 3: 필터를 적용한 Diffusion 오버샘플링

## 0. 실행 설정 / 라이브러리

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_SEEDS = list(range(10))   # 돌릴 seed
STEPS = [10]                  # 샘플링 스텝 수
GPU_ID = None                 # None이면 가장 한가한 GPU를 자동 선택. 고정하려면 "0" / "1"처럼 지정
RESUME = False                # True면 이미 결과가 있는 (method, seed)는 건너뛰고 이어서 (False면 기존 CSV를 지우고 새로 시작)
FAR_Q = 0.95                  # exp4와 같은 기준 (실제끼리 NN 거리의 95% 분위수)
RESULTS_CSV = "exp3_filtered_oversampling.csv"

if GPU_ID is None:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,utilization.gpu", "--format=csv,noheader,nounits"]).decode()
    rows = [tuple(int(v) for v in l.split(",")) for l in out.strip().splitlines()]
    GPU_ID = str(min(rows, key=lambda r: (r[1], r[2]))[0])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
print("사용 GPU:", GPU_ID)

In [ ]:
HERE = Path.cwd().resolve()
ROOT = HERE.parent  # 7_diffusion_image_filter/ (models.py, utils.py, diffusion.py, data.py)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

from models import Unet, Classifier
from utils import set_seed, DEVICE, EMA, evaluate_classifier, metrics_to_rows, append_result_rows
from diffusion import GaussianDiffusion
from data import (build_imbalanced_group_split, crop_group, CachedDataset, CachedSubset,
                  draw_from_pool, FakeQualityFilter)

## 1. 데이터 / 설정

In [ ]:
image_size, channels, batch_size, num_classes = 64, 3, 40, 3
data_root = os.environ.get("DATA_ROOT", "../../data/AM_image_data_3split")
CLASS_NAMES = ["Normal", "Underfill_50FR", "Underfill_Fan"]
train_counts = {0: 800, 1: 80, 2: 80}
test_counts = {0: 100, 1: 100, 2: 100}
timesteps = 1000
EMA_DECAY = 0.999
CLASSIFIER_EPOCHS_REF, CLASSIFIER_LR, CLASSIFIER_BETAS = 100, 2e-4, (0.5, 0.9)
POOL_PER_CLASS = 800 - 80
FILTER_FAR_Q = FAR_Q   # exp4와 같은 기준 (실제끼리 NN 거리의 95% 분위수)
GEN_BATCH = 360             # 한 번의 샘플링 호출로 생성할 장 수
EXP1_DIR = ROOT / "exp1_oversampling_step_10"
DIFFUSION_EMA_PATH = str(EXP1_DIR / "checkpoints" / "exp1_ddpm1000_diffusion_ema_seed{seed}.pt")  # exp1에서 학습한 사전학습 모델

if not RESUME and os.path.exists(RESULTS_CSV):
    os.remove(RESULTS_CSV)

transform = T.Compose([T.Resize((image_size, image_size)), T.ToTensor(), T.Lambda(lambda t: t * 2 - 1)])
full_dataset = ImageFolder(root=data_root, transform=transform)
assert full_dataset.classes == CLASS_NAMES
full_cached = CachedDataset(full_dataset)
ddpm = GaussianDiffusion(timesteps=timesteps, image_size=image_size, channels=channels)

train_dataset = test_dataset = train_loader = test_loader = None

## 2. 함수 정의

In [ ]:
def set_split(seed):
    global train_dataset, test_dataset, train_loader, test_loader
    tr, te = build_imbalanced_group_split(full_dataset, train_counts, test_counts, seed=seed)
    leaked = ({crop_group(full_dataset.samples[i][0]) for i in tr} & {crop_group(full_dataset.samples[i][0]) for i in te})
    assert not leaked, f"같은 원본이 train/test에 모두 있음: {sorted(leaked)[:5]}"
    train_dataset, test_dataset = CachedSubset(full_cached, tr), CachedSubset(full_cached, te)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=(DEVICE == "cuda"))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=(DEVICE == "cuda"))


def build_quality_filter(minority_classes=(1, 2)):
    xs, ys = zip(*[train_dataset[i] for i in range(len(train_dataset))])
    real_x, real_y = torch.stack(xs).to(DEVICE), torch.as_tensor(ys, device=DEVICE)
    qf = FakeQualityFilter(real_x, real_y, minority_classes, far_q=FILTER_FAR_Q)
    for c in minority_classes:
        print(f"  [Filter] class {c}: 거리 기준 {qf.t_far[c]:.3f} (실제 {int((real_y == c).sum())}장 기준)")
    return qf


@torch.no_grad()
def build_filtered_pool(model, qf, steps, minority_classes=(1, 2)):
    pool, rate = {}, {}
    for c in minority_classes:
        kept, n_kept, n_gen = [], 0, 0
        while n_kept < POOL_PER_CLASS:
            y = torch.full((GEN_BATCH,), c, dtype=torch.long, device=DEVICE)
            x = ddpm.sample_ddpm_respaced(model, y, DEVICE, steps=steps)
            good = x[qf.accept(x, c)]
            kept.append(good)
            n_kept += good.size(0)
            n_gen += GEN_BATCH
        pool[c] = torch.cat(kept)[:POOL_PER_CLASS]
        rate[c] = n_kept / n_gen
    return pool, rate


def train_classifier_pool(seed, pool, total_steps, tag):
    set_seed(seed)
    clf = Classifier(num_classes=num_classes, img_channels=channels, image_size=image_size).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=CLASSIFIER_LR, betas=CLASSIFIER_BETAS)
    loss_fn = nn.CrossEntropyLoss()
    clf.train()
    it, losses, t0 = iter(train_loader), [], time.time()
    for step in tqdm(range(total_steps), desc=f"[{tag} seed{seed}]", leave=False):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        xf, yf = draw_from_pool(pool, y, DEVICE)
        if xf is not None:
            x, y = torch.cat([x, xf]), torch.cat([y, yf])
        loss = loss_fn(clf(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if (step + 1) % max(1, total_steps // 5) == 0:
            tqdm.write(f"[{tag} seed{seed}] step {step+1}/{total_steps} Cls={np.mean(losses[-50:]):.4f} elapsed={time.time()-t0:.1f}s")
    return clf

## 실험 3: 필터 오버샘플링 & 평가

In [ ]:
done = set()
if RESUME and os.path.exists(RESULTS_CSV):
    _prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(_prev["method"], _prev["seed"]))

for seed in RUN_SEEDS:
    set_split(seed)
    base_steps = len(train_loader) * CLASSIFIER_EPOCHS_REF
    ema_path = DIFFUSION_EMA_PATH.format(seed=seed)
    if not os.path.exists(ema_path):
        print(f"[seed{seed}] 체크포인트 없음 → 건너뜀: {ema_path}")
        continue
    model = Unet(dim=32, channels=channels, num_classes=num_classes).to(DEVICE)
    model.load_state_dict(torch.load(ema_path, map_location=DEVICE))
    model.eval()
    print(f"\n===== Exp3 filtered oversampling | Seed {seed} (사전학습 EMA 로드: {ema_path}) =====")
    qf = build_quality_filter()
    for steps in STEPS:
        method = f"3_filtered_oversampling_ddpm1000_step{steps}"
        if (method, seed) in done:
            print(f"[{method} seed{seed}] 결과 있음 → 건너뜀"); continue
        t0 = time.time()
        set_seed(seed)
        pool, rate = build_filtered_pool(model, qf, steps)
        print(f"[step{steps} seed{seed}] 풀 생성 {time.time()-t0:.1f}초, 필터 통과율 " + ", ".join(f"class{c}: {r:.1%}" for c, r in rate.items()))
        clf = train_classifier_pool(seed, pool, base_steps, tag=f"Exp3-step{steps}")
        metrics = evaluate_classifier(clf, test_loader, DEVICE, num_classes, CLASS_NAMES)
        rows = metrics_to_rows(metrics, seed, method)
        append_result_rows(rows, RESULTS_CSV)
        print(f"[{method} seed{seed}] 총 {time.time()-t0:.1f}초")
        del pool

In [ ]:
df3 = pd.read_csv(RESULTS_CSV).drop_duplicates(subset=["method", "seed", "class"], keep="last")
df1 = pd.read_csv(EXP1_DIR / "exp1_oversampling.csv")
for d in (df1, df3):
    d["step"] = d["method"].str.extract(r"step(\d+)$")[0].astype(int)
a3 = df3.groupby(["step", "seed"])[["f1", "accuracy"]].mean().reset_index()
a1 = df1[df1["seed"].isin(a3["seed"].unique())].groupby(["step", "seed"])[["f1", "accuracy"]].mean().reset_index()
m = a1.merge(a3, on=["step", "seed"], suffixes=("_exp1", "_exp3"))
print("\n===== exp1(필터 없음) vs exp3(필터) — seed 평균 (macro f1 / accuracy) =====")
print(m.groupby("step")[["f1_exp1", "f1_exp3", "accuracy_exp1", "accuracy_exp3"]].mean().round(4))